In [1]:
from pyspark.sql import SparkSession
from datetime import datetime


# Initialize SparkSession
ss = SparkSession.builder.getOrCreate()

stations_df = ss.read.load("data/sampleData/stations.csv", format="csv", header=True, inferSchema=True, delimiter="\\t")
register_df = ss.read.load("data/sampleData/register.csv", format="csv", header=True, inferSchema=True, delimiter="\\t")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 12:25:45 WARN Utils: Your hostname, cubone, resolves to a loopback address: 127.0.1.1; using 192.168.1.113 instead (on interface wlan0)
26/05/25 12:25:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 12:25:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 12:25:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Debug
stations_df.show()

+---+---------+---------+--------------------+
| id|longitude| latitude|                name|
+---+---------+---------+--------------------+
|  1| 2.180019|41.397978|Gran Via Corts Ca...|
|  2| 2.176414|41.394381|       Plaza TetuÃ¡n|
|  3| 2.181164| 41.39375|             Ali Bei|
|  4|   2.1814|41.393364|               Ribes|
|  5| 2.180214|41.391072|  Pg LluÃ­s Companys|
|  6| 2.180508|41.391272|  Pg LluÃ­s Companys|
|  7| 2.183183|41.388867|  Pg LluÃ­s Companys|
|  8| 2.183453|41.389044|Passeig lluÃ­s co...|
|  9| 2.185294|41.385006|MarquÃ¨s de l\'Ar...|
| 10| 2.185206|41.384875|Avinguda del Marq...|
| 11| 2.183622|41.385394|             ComerÃ§|
| 12| 2.193939|41.381681|            Trelawny|
| 13| 2.195661|41.384522|pg marÃ­tim barce...|
| 14| 2.195603|41.384417|     Passeig Maritim|
| 15| 2.195706|41.386811|       Avda. Litoral|
| 16| 2.195764|41.386869|    Avinguda Litoral|
| 17| 2.170775|41.395161|              Girona|
| 18|   2.1867|41.398258|       Av. Meridiana|
| 19| 2.18669

In [3]:
# Debug
register_df.show()

+-------+-------------------+----------+----------+
|station|          timestamp|used_slots|free_slots|
+-------+-------------------+----------+----------+
|      1|2008-05-15 12:01:00|         0|        18|
|      1|2008-05-15 12:02:00|         0|        18|
|      1|2008-05-15 12:04:00|         0|        18|
|      1|2008-05-15 12:06:00|         0|        18|
|      1|2008-05-15 12:08:00|         0|        18|
|      1|2008-05-15 12:10:00|         0|        18|
|      1|2008-05-15 12:12:00|         0|        18|
|      1|2008-05-15 12:14:00|         0|        18|
|      1|2008-05-15 12:16:00|         0|        18|
|      1|2008-05-15 12:18:00|         0|        18|
|      1|2008-05-15 12:20:00|         2|        16|
|      1|2008-05-15 12:22:00|         3|        15|
|      1|2008-05-15 12:24:00|         3|        15|
|      1|2008-05-15 12:26:00|         3|        15|
|      1|2008-05-15 12:28:00|         4|        14|
|      1|2008-05-15 12:30:00|         0|        12|
|      1|200

In [4]:
register_df = register_df.filter("NOT (used_slots=0 AND free_slots=0)")

In [5]:
register_df.show()

+-------+-------------------+----------+----------+
|station|          timestamp|used_slots|free_slots|
+-------+-------------------+----------+----------+
|      1|2008-05-15 12:01:00|         0|        18|
|      1|2008-05-15 12:02:00|         0|        18|
|      1|2008-05-15 12:04:00|         0|        18|
|      1|2008-05-15 12:06:00|         0|        18|
|      1|2008-05-15 12:08:00|         0|        18|
|      1|2008-05-15 12:10:00|         0|        18|
|      1|2008-05-15 12:12:00|         0|        18|
|      1|2008-05-15 12:14:00|         0|        18|
|      1|2008-05-15 12:16:00|         0|        18|
|      1|2008-05-15 12:18:00|         0|        18|
|      1|2008-05-15 12:20:00|         2|        16|
|      1|2008-05-15 12:22:00|         3|        15|
|      1|2008-05-15 12:24:00|         3|        15|
|      1|2008-05-15 12:26:00|         3|        15|
|      1|2008-05-15 12:28:00|         4|        14|
|      1|2008-05-15 12:30:00|         0|        12|
|      1|200

In [6]:
register_df = register_df.selectExpr("station", "date_format(timestamp,'EE') as weekday", "hour(timestamp) as hour", "used_slots", "free_slots")

In [7]:
register_df.show()

+-------+-------+----+----------+----------+
|station|weekday|hour|used_slots|free_slots|
+-------+-------+----+----------+----------+
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         0|        18|
|      1|    Thu|  12|         2|        16|
|      1|    Thu|  12|         3|        15|
|      1|    Thu|  12|         3|        15|
|      1|    Thu|  12|         3|        15|
|      1|    Thu|  12|         4|        14|
|      1|    Thu|  12|         0|        12|
|      1|    Thu|  12|         4|        14|
|      1|    Thu|  12|         4|        14|
|      1|    Thu|  12|         4|        14|
|      1| 

In [8]:
critical_count_df = register_df\
    .filter("free_slots=0")\
    .groupBy("station", "weekday", "hour")\
    .agg({"free_slots":"count"})\
    .withColumnRenamed("count(free_slots)", "critical_count")

In [9]:
total_count_df = register_df\
    .groupBy("station", "weekday", "hour")\
    .count()

In [10]:
criticality_df = total_count_df\
    .join(critical_count_df, ["station", "weekday", "hour"])\
    .selectExpr("station", "weekday", "hour", "critical_count/count as criticality")

In [11]:
criticality_df.show()

+-------+-------+----+--------------------+
|station|weekday|hour|         criticality|
+-------+-------+----+--------------------+
|      1|    Wed|  14|0.018518518518518517|
|      2|    Mon|   5| 0.12477064220183487|
|      2|    Thu|  17|0.010398613518197574|
|      2|    Sun|   5|   0.127208480565371|
|      2|    Thu|  11|0.001776198934280...|
|      1|    Fri|  14|0.027491408934707903|
|      1|    Mon|  21|  0.1480836236933798|
|      2|    Thu|  13|  0.0017825311942959|
|      1|    Sat|  22| 0.12544802867383512|
|      1|    Thu|  17| 0.01386481802426343|
|      1|    Mon|   7| 0.08302583025830258|
|      1|    Wed|  19|0.003521126760563...|
|      2|    Wed|  18|0.017921146953405017|
|      1|    Sat|  20| 0.06560283687943262|
|      1|    Sat|   1| 0.15681233933161953|
|      2|    Fri|   5|  0.1348122866894198|
|      2|    Sat|   5| 0.13464991023339318|
|      2|    Sat|   2| 0.15167095115681234|
|      1|    Sat|   5| 0.25272727272727274|
|      1|    Fri|   1|  0.319148

In [12]:
threshold = 0.3

critical_register_df = criticality_df.filter(f"criticality > {threshold}") \
    .join(stations_df, criticality_df.station==stations_df.id)\
    .drop("id")\
    .withColumnRenamed("station", "station_id")\
    .withColumnRenamed("name", "address")\

critical_register_df = critical_register_df \
    .sort(critical_register_df.criticality.desc(),
          critical_register_df.station_id.asc(),
          critical_register_df.weekday.asc(), 
          critical_register_df.hour.asc())

final_output_df = critical_register_df.select(
    "station_id", 
    "weekday", 
    "hour", 
    "criticality", 
    "longitude", 
    "latitude"
)

In [13]:
final_output_df.show()

+----------+-------+----+-------------------+---------+---------+
|station_id|weekday|hour|        criticality|longitude| latitude|
+----------+-------+----+-------------------+---------+---------+
|         1|    Thu|   0| 0.4581005586592179| 2.180019|41.397978|
|         1|    Thu|   1| 0.4329608938547486| 2.180019|41.397978|
|         1|    Sun|   4|  0.403899721448468| 2.180019|41.397978|
|         1|    Wed|  23|  0.388086642599278| 2.180019|41.397978|
|         1|    Thu|   2|0.38341968911917096| 2.180019|41.397978|
|         1|    Tue|   0| 0.3743016759776536| 2.180019|41.397978|
|         1|    Wed|  22|0.37122557726465366| 2.180019|41.397978|
|         1|    Mon|   1| 0.3659217877094972| 2.180019|41.397978|
|         1|    Wed|   0|0.34108527131782945| 2.180019|41.397978|
|         1|    Mon|   0| 0.3380281690140845| 2.180019|41.397978|
|         1|    Tue|   1| 0.3352272727272727| 2.180019|41.397978|
|         1|    Fri|   0| 0.3307291666666667| 2.180019|41.397978|
|         

In [14]:
final_output_df.write.csv("output/", header=True)